In [1]:
import sys
import json
import requests
import pandas as pd

WEBSITE_API = "https://rest.uniprot.org/"

def get_url(url, **kwargs):
    """Fetch a URL and return the response, exiting on an HTTP error."""
    response = requests.get(url, **kwargs)

    if not response.ok:
        print(response.text)
        response.raise_for_status()
        sys.exit()

    return response

# 1. Data layout

# Surfactant protein D ("Sftpd") is the knocked-out gene, so it is left out when
# ranking genes in the knockout comparisons.

# The data file is referenced by NAME. Put the CSV in the same folder as this
# notebook (or launch Jupyter from that folder) and it will be found. Change
# CSV_FILE below if your file is named differently.

CSV_FILE = "NewAffydata.csv"           # <-- the CSV file name

raw = pd.read_csv(CSV_FILE, header=None, dtype=str)
print(f"Loading data from: {CSV_FILE}")

LABEL_COL = 0                          # first column holds the condition labels
EXCLUDE_NAMES = {"sftpd"}              # surfactant protein D (knocked out), skipped when ranking
TOP_N = 3                              # number of top genes to report per condition

DATA_COLS = list(range(1, raw.shape[1]))

gene_names = raw.loc[0, DATA_COLS].tolist()

# 2. Map each gene to a single column

# Several genes appear in more than one probe column; the first column for each
# gene is used. A combined-probe label such as "Ear1 /// Ear12 /// Ear2 /// Ear3"
# maps one probe to several genes, and the first symbol before "///" is used for
# the UniProt search.
def search_symbol(raw_name):
    """First symbol before any '///' in a combined-probe label."""
    return raw_name.split("///")[0].strip()

gene_positions = {}                    # gene symbol -> its first data column
gene_query = {}                        # gene symbol -> symbol used for the UniProt search
for pos, name in enumerate(gene_names):
    clean = str(name).strip()
    if clean in gene_positions:
        continue                       # keep only the first column for this gene
    gene_positions[clean] = DATA_COLS[pos]
    gene_query[clean] = search_symbol(clean)

# 3. Read each condition row into a per-gene value

def row_values(line):
    """Value of each gene's column for one row of the file."""
    numeric = pd.to_numeric(raw.loc[line, DATA_COLS], errors="coerce")
    return {gene: numeric[col] for gene, col in gene_positions.items()}

row_label = {}                          # row index -> condition label
row_genes = {}                          # row index -> {gene: value}
for line in range(1, raw.shape[0]):     # every row after the header row
    label = raw.loc[line, LABEL_COL]
    if pd.isna(label):
        continue
    row_label[line] = str(label).strip()
    row_genes[line] = row_values(line)

# 4. Split each label into (strain, condition)

STRAINS = ["BL/6", "Balb/c", "SP-D KO"]

def split_label(label):
    for strain in STRAINS:
        if label.startswith(strain):
            return strain, label[len(strain):].strip()
    return None, label

# For each strain, map its conditions to their row indices.
rows_by_strain = {s: {} for s in STRAINS}
for line, label in row_label.items():
    strain, cond = split_label(label)
    if strain is not None:
        rows_by_strain[strain][cond] = line

# 5. Look up a gene symbol on UniProt (Mus musculus)

_uniprot_cache = {}

def lookup_uniprot(symbol):
    """Return (accession, entry URL) for a mouse gene symbol, caching results."""
    if symbol in _uniprot_cache:
        return _uniprot_cache[symbol]

    # gene:<symbol> is an indexed lookup; size=1 returns only the best hit.
    query_url = (
        f"{WEBSITE_API}uniprotkb/search"
        f"?query=gene:{symbol}+AND+organism_id:10090&fields=accession&format=json&size=1"
    )
    r = get_url(query_url)
    results = r.json()["results"]

    if results:
        accession = results[0]["primaryAccession"]
        entry_url = f"https://www.uniprot.org/uniprotkb/{accession}/entry"
    else:
        accession, entry_url = "NA", "NA"

    _uniprot_cache[symbol] = (accession, entry_url)
    return accession, entry_url

# 6. Build a comparison table for two strains

def build_table(strain_a, strain_b, exclude_spd=True):
    """Compare strain_a with strain_b; the difference is strain_b minus strain_a.

    When exclude_spd is True, surfactant protein D is left out of the ranking.
    """
    a_rows = rows_by_strain[strain_a]
    b_rows = rows_by_strain[strain_b]
    conditions = [c for c in a_rows if c in b_rows]   # conditions shared by both strains

    diff_header = f"Difference ([{strain_b}] - [{strain_a}])"

    records = []
    for cond in conditions:
        a_vals = row_genes[a_rows[cond]]
        b_vals = row_genes[b_rows[cond]]

        # Difference for each gene, as strain_b minus strain_a.
        diffs = {
            gene: b_vals[gene] - a_vals[gene]
            for gene in gene_positions
            if not (exclude_spd and gene.lower() in EXCLUDE_NAMES)
        }

        # Take the TOP_N genes with the largest absolute difference.
        working = dict(diffs)
        top_genes = []
        for _ in range(TOP_N):
            gene = max(working, key=lambda g: abs(working[g]))
            top_genes.append(gene)
            del working[gene]                          # remove it so the next pass finds the next largest

        gene_labels, differences, accessions, links = [], [], [], []
        for gene in top_genes:
            gene_labels.append(gene)
            differences.append(round(float(diffs[gene]), 3))
            accession, entry_url = lookup_uniprot(gene_query[gene])
            accessions.append(accession)
            links.append(entry_url)

        records.append(
            {
                "Condition": row_label[a_rows[cond]],
                "Gene Names": gene_labels,
                diff_header: differences,
                "Accession Numbers": accessions,
                "UniProt Links": links,
            }
        )

    return pd.DataFrame(records)

# 7. Build and display the three tables

table_bl6 = build_table("BL/6", "SP-D KO")
table_balb = build_table("Balb/c", "SP-D KO")

from IPython.display import display

# Show full cell contents so the UniProt links are not truncated.
pd.set_option("display.max_colwidth", None)

def show(df, title):
    """Display a table with left-aligned columns."""
    print(title)
    try:
        display(df.style.set_properties(**{"text-align": "left"})
                        .set_table_styles([{"selector": "th", "props": [("text-align", "left")]}]))
    except Exception:
        print(df.to_string(index=False, justify="left"))

show(table_bl6, "BL/6 vs SP-D KO  (top 3 genes per condition)")
show(table_balb, "Balb/c vs SP-D KO  (top 3 genes per condition)")

Loading data from: NewAffydata.csv
BL/6 vs SP-D KO  (top 3 genes per condition)


,Condition,Gene Names,Difference ([SP-D KO] - [BL/6]),Accession Numbers,UniProt Links
0,BL/6 air,"['Ear1', 'Ear3', 'Srrm2']","[1779.0, 489.0, 447.0]","['P97426', 'O35290', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/O35290/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
1,BL/6 24h Af+air,"['Nos1ap', 'Prmt2', 'C2']","[-1904.0, -1429.0, -807.0]","['B7ZNH1', 'F6Y3M9', 'B8JJM9']","['https://www.uniprot.org/uniprotkb/B7ZNH1/entry', 'https://www.uniprot.org/uniprotkb/F6Y3M9/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry']"
2,BL/6 168h Af+air,"['Avpi1', 'C1qa', 'Ear1']","[3461.0, 3207.0, 2149.0]","['E0CXY9', 'Q3TXB1', 'P97426']","['https://www.uniprot.org/uniprotkb/E0CXY9/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry']"
3,BL/6 O3,"['Ear1', 'C2', 'Srrm2']","[2498.0, -1120.0, 892.0]","['P97426', 'B8JJM9', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
4,BL/6 24h Af+O3,"['Ear1', 'C1qc', 'Avpi1']","[-2042.0, 1484.0, -1440.0]","['P97426', 'Q02105', 'E0CXY9']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/Q02105/entry', 'https://www.uniprot.org/uniprotkb/E0CXY9/entry']"
5,BL/6 168h Af+O3,"['C1qa', 'C2', 'Ptpla']","[7415.0, 2565.0, -2171.0]","['Q3TXB1', 'B8JJM9', 'Q3V4A5']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/Q3V4A5/entry']"


Balb/c vs SP-D KO  (top 3 genes per condition)


,Condition,Gene Names,Difference ([SP-D KO] - [Balb/c]),Accession Numbers,UniProt Links
0,Balb/c air,"['Ear1', 'Padi2', 'Ear3']","[1750.0, -592.0, 478.0]","['P97426', 'A3KME9', 'O35290']","['https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/A3KME9/entry', 'https://www.uniprot.org/uniprotkb/O35290/entry']"
1,Balb/c 24h Af+air,"['C1qa', 'Ptpla', 'Srrm2']","[-3526.0, -2580.0, -2425.0]","['Q3TXB1', 'Q3V4A5', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/Q3V4A5/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
2,Balb/c 168h Af+air,"['Avpi1', 'Ear1', 'C1qa']","[3239.0, 1768.0, 1443.0]","['E0CXY9', 'P97426', 'Q3TXB1']","['https://www.uniprot.org/uniprotkb/E0CXY9/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry']"
3,Balb/c O3,"['C1qa', 'Ear1', 'Srrm2']","[-3486.0, 2536.0, 1401.0]","['Q3TXB1', 'P97426', 'A0A087WPS9']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry', 'https://www.uniprot.org/uniprotkb/A0A087WPS9/entry']"
4,Balb/c 24h Af+O3,"['C2', 'C1s /// LOC100044326', 'C1qa']","[-2870.0, -2123.0, -1930.0]","['B8JJM9', 'Q14DT6', 'Q3TXB1']","['https://www.uniprot.org/uniprotkb/B8JJM9/entry', 'https://www.uniprot.org/uniprotkb/Q14DT6/entry', 'https://www.uniprot.org/uniprotkb/Q3TXB1/entry']"
5,Balb/c 168h Af+O3,"['C1qa', 'Nos1ap', 'Ear1']","[5143.0, 2305.0, 1796.0]","['Q3TXB1', 'B7ZNH1', 'P97426']","['https://www.uniprot.org/uniprotkb/Q3TXB1/entry', 'https://www.uniprot.org/uniprotkb/B7ZNH1/entry', 'https://www.uniprot.org/uniprotkb/P97426/entry']"


In [3]:
import time

# --- endpoints / settings ----------------------------------------------------
PSICQUIC_INTACT = ("https://www.ebi.ac.uk/Tools/webservices/psicquic/"
                   "intact/webservices/current/search/interactor")
STRING_API       = "https://string-db.org/api"
MOUSE_TAXON      = 10090                        # Mus musculus
CALLER_ID        = "affy-spd-interaction-notebook"
TOP_INTERACTIONS = 30                           # interactions to report per database

# 8a. Collect the identified genes (symbol -> accession) from the KO tables

def collect_identified_genes(*tables):
    """Gather every gene reported in the comparison tables as {symbol: accession}."""
    genes = {}
    for table in tables:
        for _, row in table.iterrows():
            for symbol, accession in zip(row["Gene Names"], row["Accession Numbers"]):
                if accession and accession != "NA":
                    genes.setdefault(symbol, accession)
    return genes

identified_genes = collect_identified_genes(table_bl6, table_balb)
identified_symbols = sorted(identified_genes)

print("Identified genes carried into the interaction search:")
for symbol in identified_symbols:
    print(f"  {symbol:10s} {identified_genes[symbol]}")
print()

# 8b. MITAB 2.7 parsing helpers (shared by both databases)

def _mitab_label(field):
    """'psi-mi:"MI:0018"(two hybrid)' -> 'two hybrid'."""
    if not field or field == "-":
        return "NA"
    first = field.split("|")[0]
    if "(" in first and first.rstrip().endswith(")"):
        return first[first.rindex("(") + 1: first.rindex(")")].strip()
    return first.strip()

def _mitab_accession(field):
    """'uniprotkb:P12345' -> 'P12345'."""
    if not field or field == "-":
        return None
    token = field.split("|")[0].replace('"', "")
    return token.split(":")[-1].split("-")[0].strip()

def _mitab_pub_count(field):
    if not field or field == "-":
        return 0
    return len([p for p in field.split("|") if p.strip()])

def _mitab_miscore(field):
    if not field or field == "-":
        return 0.0
    for part in field.split("|"):
        if "intact-miscore:" in part:
            try:
                return float(part.split("intact-miscore:")[1].split("|")[0])
            except (ValueError, IndexError):
                pass
    return 0.0

def _mitab_aliases(field):
    """Lower-cased text of an alias column, for gene-name matching."""
    return (field or "").lower()

# 8c. IntAct query (PSICQUIC -> MITAB 2.7)

def intact_rows_for(accession):
    """Return parsed IntAct MITAB rows for one UniProt accession."""
    url = f"{PSICQUIC_INTACT}/{accession}"
    try:
        r = requests.get(url, params={"format": "tab27"}, timeout=60)
    except Exception as exc:
        print(f"  [IntAct] {accession}: request failed ({exc})")
        return []
    if not r.ok:
        print(f"  [IntAct] {accession}: HTTP {r.status_code}")
        return []
    if not r.text.strip():
        return []

    rows = []
    for line in r.text.strip().split("\n"):
        cols = line.split("\t")
        if len(cols) < 15:
            continue
        # 0 idA, 1 idB, 4 aliasA, 5 aliasB, 6 detection method,
        # 8 publications, 11 interaction type, 14 confidence
        rows.append({
            "accA": _mitab_accession(cols[0]),
            "accB": _mitab_accession(cols[1]),
            "aliasA": _mitab_aliases(cols[4]),
            "aliasB": _mitab_aliases(cols[5]),
            "assay": _mitab_label(cols[6]),
            "interaction_type": _mitab_label(cols[11]),
            "publications": _mitab_pub_count(cols[8]),
            "miscore": _mitab_miscore(cols[14]),
        })
    time.sleep(0.3)
    return rows

# Cache IntAct rows per accession (reused for the STRING assay lookup).
_intact_cache = {}

def intact_rows_cached(accession):
    if accession not in _intact_cache:
        _intact_cache[accession] = intact_rows_for(accession)
    return _intact_cache[accession]

def build_intact_table(genes, top_n=TOP_INTERACTIONS):
    """Top IntAct interactions for `genes` (symbol -> accession), by MI score."""
    acc_to_symbol = {acc: sym for sym, acc in genes.items()}
    records, seen = [], set()

    for symbol, accession in genes.items():
        rows = intact_rows_cached(accession)
        if not rows:
            print(f"  (no IntAct interactions found for {symbol} / {accession})")
        for row in rows:
            accA, accB = row["accA"], row["accB"]
            if accession not in (accA, accB):
                continue
            partner_acc = accB if accA == accession else accA
            if not partner_acc or partner_acc == accession:
                continue
            pair = tuple(sorted((accession, partner_acc)))
            if pair in seen:
                continue
            seen.add(pair)
            records.append({
                "Gene A": symbol,
                "Gene B": acc_to_symbol.get(partner_acc, partner_acc),
                "Accession A": accession,
                "Accession B": partner_acc,
                "Within identified set": partner_acc in acc_to_symbol,
                "Interaction Type": row["interaction_type"],
                "Assay Type (detection method)": row["assay"],
                "Publications": row["publications"],
                "IntAct MI score": round(row["miscore"], 3),
            })

    df = pd.DataFrame(records)
    if not df.empty:
        df = (df.sort_values("IntAct MI score", ascending=False)
                .head(top_n).reset_index(drop=True))
    return df

# 8d. STRING query (network) + IntAct assay enrichment

STRING_CHANNELS = {                             # STRING field -> evidence channel
    "escore": "experiments",
    "dscore": "curated databases",
    "ascore": "coexpression",
    "tscore": "textmining",
    "nscore": "gene neighborhood",
    "fscore": "gene fusion",
    "pscore": "phylogenetic co-occurrence",
}

def _clean_symbol(symbol):
    """Drop any combined-probe suffix so STRING sees a single gene symbol."""
    return symbol.split("///")[0].strip()

def string_map_ids(symbols):
    """Map gene symbols to STRING identifiers -> {symbol: stringId}."""
    params = {
        "identifiers": "\r".join(_clean_symbol(s) for s in symbols),
        "species": MOUSE_TAXON,
        "limit": 1,
        "echo_query": 1,
        "caller_identity": CALLER_ID,
    }
    r = requests.post(f"{STRING_API}/json/get_string_ids", data=params, timeout=30)
    r.raise_for_status()
    clean_to_symbol = {_clean_symbol(s): s for s in symbols}
    mapping = {}
    for row in r.json():
        queried = row.get("queryItem", row.get("preferredName"))
        mapping[clean_to_symbol.get(queried, queried)] = row["stringId"]
    return mapping

def intact_assay_for_pair(gene_a, gene_b, genes):
    """Look up the IntAct assay(s) for the interaction between two gene symbols.

    Uses the cached IntAct rows for gene_a's accession (if it is an identified
    gene) and scans them for gene_b by name; returns 'not in IntAct' if none.
    """
    acc_a = genes.get(gene_a)
    if not acc_a:
        return "not in IntAct"
    b_low = gene_b.lower()
    methods = []
    for row in intact_rows_cached(acc_a):
        text = f"{row['aliasA']} {row['aliasB']} {row['accA']} {row['accB']}".lower()
        if b_low in text and row["assay"] != "NA":
            methods.append(row["assay"])
    if methods:
        return "; ".join(sorted(set(methods)))
    return "not in IntAct"

def build_string_table(symbols, genes, top_n=TOP_INTERACTIONS):
    """Top STRING interactions (by combined confidence), enriched with IntAct assay."""
    if len(symbols) < 2:
        return pd.DataFrame()

    string_ids = string_map_ids(symbols)
    id_to_symbol = {sid: sym for sym, sid in string_ids.items()}
    identified_string_names = {_clean_symbol(s) for s in symbols}
    if len(string_ids) < 2:
        return pd.DataFrame()

    params = {
        "identifiers": "\r".join(string_ids.values()),
        "species": MOUSE_TAXON,
        "add_nodes": top_n,          # widen the network so >= top_n interactions exist
        "caller_identity": CALLER_ID,
    }
    r = requests.post(f"{STRING_API}/json/network", data=params, timeout=30)
    r.raise_for_status()

    records, seen = [], set()
    for row in r.json():
        a = row.get("preferredName_A") or id_to_symbol.get(row.get("stringId_A"), row.get("stringId_A"))
        b = row.get("preferredName_B") or id_to_symbol.get(row.get("stringId_B"), row.get("stringId_B"))
        pair = tuple(sorted((a, b)))
        if pair in seen:
            continue
        seen.add(pair)

        channels = {name: float(row.get(field, 0) or 0)
                    for field, name in STRING_CHANNELS.items()}
        best_channel = max(channels, key=channels.get)
        both_identified = (a in identified_string_names) and (b in identified_string_names)

        records.append({
            "Gene A": a,
            "Gene B": b,
            "Within identified set": both_identified,
            "STRING evidence channel": best_channel,
            "STRING confidence score": round(float(row.get("score", 0) or 0), 3),
        })

    df = pd.DataFrame(records)
    if df.empty:
        return df

    df = (df.sort_values("STRING confidence score", ascending=False)
            .head(top_n).reset_index(drop=True))

    # enrich the kept interactions with the real assay type from IntAct
    df["Assay Type (from IntAct)"] = [
        intact_assay_for_pair(a, b, genes) for a, b in zip(df["Gene A"], df["Gene B"])
    ]
    df = df[["Gene A", "Gene B", "Within identified set",
             "Assay Type (from IntAct)",
             "STRING evidence channel", "STRING confidence score"]]
    return df

# 8e. Run both searches and display

print(f"Querying EMBL-EBI (IntAct) for the top {TOP_INTERACTIONS} interactions...\n")
intact_table = build_intact_table(identified_genes)

print(f"\nQuerying STRING-DB for the top {TOP_INTERACTIONS} interactions...\n")
try:
    string_table = build_string_table(identified_symbols, identified_genes)
except Exception as exc:
    print(f"  [STRING] search failed: {exc}")
    string_table = pd.DataFrame()

# IntAct table
if intact_table.empty:
    print("\nNo IntAct interactions were found for any of the identified genes.")
    print("These mouse proteins may have no curated IntAct records, or the EBI")
    print("service was unreachable (see the messages above).")
else:
    print(f"\nIntAct: found {len(intact_table)} interaction(s).")
    show(intact_table,
         f"EMBL-EBI (IntAct): top {len(intact_table)} interactions of the identified genes")

# STRING table
if string_table.empty:
    print("\nNo STRING interactions were returned (or the STRING service was unreachable).")
else:
    print(f"\nSTRING: found {len(string_table)} interaction(s).")
    show(string_table,
         f"STRING-DB: top {len(string_table)} interactions of the identified genes")

Identified genes carried into the interaction search:
  Avpi1      E0CXY9
  C1qa       Q3TXB1
  C1qc       Q02105
  C1s /// LOC100044326 Q14DT6
  C2         B8JJM9
  Ear1       P97426
  Ear3       O35290
  Nos1ap     B7ZNH1
  Padi2      A3KME9
  Prmt2      F6Y3M9
  Ptpla      Q3V4A5
  Srrm2      A0A087WPS9

Querying EMBL-EBI (IntAct) for the top 30 interactions...

  (no IntAct interactions found for Ear1 / P97426)
  (no IntAct interactions found for Ear3 / O35290)
  (no IntAct interactions found for Srrm2 / A0A087WPS9)
  (no IntAct interactions found for Nos1ap / B7ZNH1)
  (no IntAct interactions found for Prmt2 / F6Y3M9)
  (no IntAct interactions found for C2 / B8JJM9)
  (no IntAct interactions found for Avpi1 / E0CXY9)
  (no IntAct interactions found for C1qa / Q3TXB1)
  (no IntAct interactions found for Ptpla / Q3V4A5)
  (no IntAct interactions found for Padi2 / A3KME9)
  (no IntAct interactions found for C1s /// LOC100044326 / Q14DT6)

Querying STRING-DB for the top 30 interaction

,Gene A,Gene B,Accession A,Accession B,Within identified set,Interaction Type,Assay Type (detection method),Publications,IntAct MI score
0,C1qc,E9Q401,Q02105,E9Q401,False,association,anti bait coimmunoprecipitation,2,0.350000



STRING: found 30 interaction(s).
STRING-DB: top 30 interactions of the identified genes


,Gene A,Gene B,Within identified set,Assay Type (from IntAct),STRING evidence channel,STRING confidence score
0,C1rb,C1s2,False,not in IntAct,textmining,0.999000
1,Serping1,C1s2,False,not in IntAct,textmining,0.999000
2,C1ra,C1s2,False,not in IntAct,textmining,0.999000
3,C1ra,C1s1,False,not in IntAct,textmining,0.999000
4,Serping1,C1rb,False,not in IntAct,textmining,0.999000
5,Serping1,C1ra,False,not in IntAct,textmining,0.999000
6,Serping1,C1s1,False,not in IntAct,textmining,0.999000
7,C1qb,C1qa,False,not in IntAct,textmining,0.999000
8,C1qc,C1qb,False,not in IntAct,textmining,0.999000
9,C1qc,C1qa,True,not in IntAct,coexpression,0.999000
